# COCO Format

> converting dataset into coco format

In [ ]:
#| default_exp preprocessing.coco_format

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import cv2
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import json
from argparse import ArgumentParser
from fastcore.all import *
import os
import shutil

### Data Path

In [ ]:
MAT='47834'
im_path = Path(fr'/home/hpc_storage/data/projects/userB/data/cegled_econo/{MAT}/new_16bit_processed')

DST_OVERLAY =im_path.parent/f"curated8bit/void_mask_overlay_{MAT}" 
DST_MSK = im_path.parent/f"curated8bit/chip_mask_{MAT}"
DST_IMG  = im_path.parent/f"curated8bit/chip_img_{MAT}"

In [ ]:
DST_OVERLAY.ls()

In [ ]:
trn_p = int(len(DST_OVERLAY.ls()) * .7)
trn_idx =[i for i in range(trn_p)]
trn_idx[::-1]
val_idx = [i for i in range(len(DST_OVERLAY.ls())) if i not in trn_idx]
len(trn_idx), len(val_idx)


In [ ]:
TRN_IM_DIR = DST_OVERLAY.parent/f"train/images"
TRN_MSK_DIR = DST_OVERLAY.parent/f"train/masks"
VAL_IM_DIR = DST_OVERLAY.parent/f"test/images"
VAL_MSK_DIR = DST_OVERLAY.parent/f"test/masks"


In [ ]:
L([TRN_IM_DIR, TRN_MSK_DIR, VAL_IM_DIR, VAL_MSK_DIR]).map(lambda x: Path(x).mkdir(parents=True, exist_ok=True))

In [ ]:
for i, img in enumerate(DST_OVERLAY.ls()):
    if i in trn_idx:
        shutil.copy(Path(DST_IMG,img.name), TRN_IM_DIR)
        shutil.copy(Path(DST_MSK,img.name), TRN_MSK_DIR)
    else:
        shutil.copy(Path(DST_IMG,img.name), VAL_IM_DIR)
        shutil.copy(Path(DST_MSK,img.name), VAL_MSK_DIR)

In [ ]:
data_path = Path('/home/hpc_storage/data/projects/userA/sam3_data/masks/8_In_18_0_1.png')
data_path.exists()

In [ ]:
mask = cv2.imread(data_path, cv2.IMREAD_GRAYSCALE)


In [ ]:
mask.size, mask.dtype, mask.shape

In [ ]:
_, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY) #ensure binary

In [ ]:
mask 

In [ ]:
contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE) #find contours

In [ ]:
contours

In [ ]:
annotations = []
for contour in contours:
    contour = contour.flatten().tolist()
    if len(contour) < 6:  # Need at least 3 points to form a polygon
        continue
    x, y, w, h = cv2.boundingRect(np.array(contour).reshape(-1,1,2))
    area = cv2.contourArea(np.array(contour).reshape(-1, 1, 2))
    
    annotation = {
        "id": len(annotations) + 1,
        "image_id": 1, 
        "category_id": 1,
        "segmentation": [contour],
        "area": area,
        "bbox": [x, y, w, h],
        "iscrowd": 0
    }
    annotations.append(annotation)

In [ ]:
annotations

In [ ]:
#| export
def mask_to_coco (mask, img_id, category_id,annotation_id):
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE) #find contours
    annotations = []

    for contour in contours:
        contour = contour.flatten().tolist()
        if len(contour) < 6:  # Need at least 3 points to form a polygon
            continue
        x, y, w, h = cv2.boundingRect(np.array(contour).reshape(-1,1,2))
        area = cv2.contourArea(np.array(contour).reshape(-1, 1, 2))
    
        annotation = {
            "id": annotation_id,
            "image_id": img_id, 
            "category_id": category_id,
            "segmentation": [contour],
            "area": area,
            "bbox": [x, y, w, h],
            "iscrowd": 0
        }
        annotations.append(annotation)
        annotation_id += 1

    return annotations


#### Why `mask_to_coco` uses the passed `annotation_id`

`mask_to_coco` converts a binary mask into COCO annotation dicts (one per external
contour). Each annotation needs a **globally unique** `id`, because pycocotools'
`COCO.createIndex()` keys annotations by `id` — colliding ids silently overwrite each
other and corrupt evaluation.

The previous version set `"id": len(annotations) + 1`, which restarted at `1` for every
image, so ids collided across images. `form_coco_dataset` already threads a running
`annotation_id` and advances it by `len(annotations)` per image, so the fix is simply to
stamp each annotation with the passed `annotation_id` and increment it per contour.

The test below verifies that, across two images processed with a shared running counter,
every produced annotation id is unique.


In [ ]:
import tempfile

def _make_two_blob_mask(h: int = 40, w: int = 40) -> np.ndarray:
    "Binary mask with two separated square blobs -> two external contours."
    m = np.zeros((h, w), dtype=np.uint8)
    m[5:15, 5:15] = 255
    m[25:35, 25:35] = 255
    return m

with tempfile.TemporaryDirectory() as _tmp:
    # Happy path: two images, shared running annotation_id counter (as form_coco_dataset does)
    mask_a = _make_two_blob_mask()
    mask_b = _make_two_blob_mask()

    ann_id = 1
    anns_a = mask_to_coco(mask_a, img_id=1, category_id=1, annotation_id=ann_id)
    ann_id += len(anns_a)
    anns_b = mask_to_coco(mask_b, img_id=2, category_id=1, annotation_id=ann_id)
    ann_id += len(anns_b)

    all_ids = [a["id"] for a in anns_a + anns_b]
    unique = len(all_ids) == len(set(all_ids))
    print(f"{'✓' if unique else '✗'} globally unique ids across images: {all_ids}")
    assert unique, "annotation ids collided across images"
    assert all(a["image_id"] == 1 for a in anns_a) and all(a["image_id"] == 2 for a in anns_b)

    # Edge case: empty mask -> no annotations, counter unaffected
    empty = np.zeros((40, 40), dtype=np.uint8)
    anns_empty = mask_to_coco(empty, img_id=3, category_id=1, annotation_id=ann_id)
    print(f"{'✓' if anns_empty == [] else '✗'} empty mask yields no annotations: {anns_empty}")
    assert anns_empty == []

    print("status=PASS")


In [ ]:
mask_dir = Path('/home/hpc_storage/data/projects/userA/sam3_data/masks/')
img_dir = Path('/home/hpc_storage/data/projects/userA/sam3_data/images/')


In [ ]:
len(list(img_dir.glob('*.png')))

In [ ]:
import os

In [ ]:
mask_path = list(mask_dir.glob('*.png'))[0]
base = mask_path.stem
base

In [ ]:
categories = [
    {"id": 1, "name": "pin"}
]

In [ ]:
coco_dataset = {
    "images": [],
    "annotations": [],
    "categories": categories
}
coco_dataset

In [ ]:
base = mask_path.stem
img_path = img_dir / f"{base}.png"
img_path

In [ ]:
img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
height, width = img.shape[:2]
img.shape

In [ ]:
coco_dataset["images"].append({
        "id": 1,
        "file_name": img_path.name,
        "width": width,
        "height": height
    })
coco_dataset

In [ ]:
annotations = mask_to_coco(mask, img_id=1, category_id=1, annotation_id=1)
coco_dataset["annotations"].extend(annotations)
coco_dataset

In [ ]:
for img_id, mask_path in enumerate(mask_dir.glob('*.png'), start=1):
    if img_id >15:
        break
    print(img_id, mask_path)

In [ ]:
categories

In [ ]:

categories_name_to_id = {c["name"]:c["id"] for c in categories}
categories_name_to_id

In [ ]:
cat = [
    {"id": 1, "name": "images"},
    {"id": 2, "name": "masks"},
    {"id": 3, "name": "pins"}
]

categories_name_to_id = {c["name"]:c["id"] for c in cat}
categories_name_to_id

In [ ]:
#if we have multiple categories of img (arranged into subfolders with catgeory names in root mask folder)
data_dir = Path('/home/hpc_storage/data/projects/userA/sam3_data/')
for category_folder in os.listdir(data_dir):
    subfolder = data_dir / category_folder
    if not subfolder.is_dir():
        continue
    if category_folder not in categories_name_to_id:
        print(f"Skipping unknown category folder: {category_folder}")
        continue
    category_id = categories_name_to_id[category_folder]
category_id

In [ ]:
#| export
def form_coco_dataset(mask_dir, img_dir, categories):
    coco_dataset = {
        "info": {
            "description": "Eberhard Pins Dataset",
            "version": "1.0",
            "year": 2025
        },
        "images": [],
        "annotations": [],
        "categories": categories
    }

    annotation_id = 1

    for img_id, mask_path in tqdm(enumerate(mask_dir.glob('*.png'), start=1)):
        #if img_id >20:
        #    break
        base = mask_path.stem
        img_path = img_dir / f"{base}.png"
    
        # Read image to get dimensions
        img = cv2.imread(str(img_path))
        height, width = img.shape[:2]
    
        # Add image info to coco dataset
        coco_dataset["images"].append({
            "id": img_id,
            "file_name": img_path.name,
            "width": width,
            "height": height
        })
    
        # Read mask
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY) #ensure binary
    
        # Get annotations from mask
        annotations = mask_to_coco(mask, img_id, category_id=2, annotation_id=annotation_id)
    
        # Update annotation_id for next iteration
        annotation_id += len(annotations)
    
        # Add annotations to coco dataset
        coco_dataset["annotations"].extend(annotations)
    
    return coco_dataset

In [ ]:
categories = [
    {"id": 1, "name": "pin"}
]
mask_dir = Path('/home/hpc_storage/data/projects/userA/sam3_data/masks/')
img_dir = Path('/home/hpc_storage/data/projects/userA/sam3_data/images/')
coco_data = form_coco_dataset(mask_dir, img_dir, categories)
coco_data

In [ ]:
!pwd

In [ ]:
#| export
def save_ann_file (output_dir,file_name, coco_data):
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, f"{file_name}.json")
    with open(output_path,'w') as f:
        json.dump(coco_data,f,indent=2)

In [ ]:
#| export
def build_coco_metadata(coco_data: dict) -> dict:
    """Return a metadata summary for a COCO dataset.

    Reports total image/annotation/category counts and, for each category,
    the number of unique images that contain at least one annotation of that
    category and the total number of annotations.
    """
    images = coco_data.get("images", [])
    annotations = coco_data.get("annotations", [])
    categories = coco_data.get("categories", [])

    ann_counts = {}
    img_ids_per_cat = {}
    for ann in annotations:
        cat_id = ann.get("category_id")
        ann_counts[cat_id] = ann_counts.get(cat_id, 0) + 1
        img_ids_per_cat.setdefault(cat_id, set()).add(ann.get("image_id"))

    per_category = []
    for cat in categories:
        cat_id = cat["id"]
        per_category.append({
            "id": cat_id,
            "name": cat.get("name", ""),
            "image_count": len(img_ids_per_cat.get(cat_id, set())),
            "annotation_count": ann_counts.get(cat_id, 0),
        })

    return {
        "total_images": len(images),
        "total_annotations": len(annotations),
        "total_categories": len(categories),
        "categories": per_category,
    }


#| export
def save_metadata_file(output_dir, file_name, coco_data):
    """Write a `{file_name}_metadata.json` summary next to the COCO JSON."""
    os.makedirs(output_dir, exist_ok=True)
    metadata = build_coco_metadata(coco_data)
    output_path = os.path.join(output_dir, f"{file_name}_metadata.json")
    with open(output_path, "w") as f:
        json.dump(metadata, f, indent=2)


In [ ]:
#| export
def main_():
    parser = ArgumentParser()
    parser.add_argument('--img_dir', type=str,  help='Path to the input images', default='/home/hpc_storage/data/projects/userA/sam3_data/images')
    parser.add_argument('--mask_dir', type=str, help='Path to the masks', default='/home/hpc_storage/data/projects/userA/sam3_data/masks')
    #parser.add_argument('--categories', type=str, help='list of category dict (e.g. [{"id":1 "name":"pin"}])')
    parser.add_argument('--output_dir', type=str, help='Path to save the output annotated file', default='/home/hpc_storage/data/projects/userA/sam3_data/annotation')
    parser.add_argument('--file_name', type = str, help='Name of the json annotation file', default='annotation')
    args = parser.parse_args()
    return args

In [ ]:
#| export
if __name__ == '__main__':
    parser_ = main_()
    img_dir = Path(parser_.img_dir)
    mask_dir = Path(parser_.mask_dir)
    #categories = json.loads(parser_.categories)
    output_dir = Path(parser_.output_dir)
    file_name = parser_.file_name

    coco_data = form_coco_dataset(mask_dir, img_dir, categories= [{"id": 2, "name": "eberhard_pin"}])
    save_ann_file(output_dir, file_name, coco_data)
    save_metadata_file(output_dir, file_name, coco_data)


In [ ]:
categories = [
    {"id": 1, "name": "pin"}
]
mask_dir = Path('/home/hpc_storage/data/projects/userA/sam3_data/masks/')
img_dir = Path('/home/hpc_storage/data/projects/userA/sam3_data/images/')
file_name = 'annotation'
output_dir = Path('/home/hpc_storage/data/projects/userA/sam3_data/annotation')

In [ ]:
CURRENT_NB='/home/hpc_storage/data/projects/userA/sam3_fine_tune/nbs'
import os
os.chdir(CURRENT_NB)

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
def foo(): pass

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()